# Part 3: NLP and Sequence Modeling Mini Project
**Dataset:** Customer Review Sentiment Dataset (part_3_nlp_sequence_modeling)
**Problem Type:** Text Classification — Sentiment Analysis (Positive / Negative / Neutral)
**Dataset Source:** https://drive.google.com/drive/folders/1akV6po4Nrgkc3yQrJkzA6cJlV-wBvUYs?usp=sharing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

os.makedirs('results', exist_ok=True)
print('Libraries loaded. TensorFlow:', tf.__version__)

## Task 1: Dataset Understanding

In [ ]:
np.random.seed(42)

positive_templates = [
    'This product is absolutely amazing and works perfectly.',
    'Great quality, fast delivery, highly recommend to everyone.',
    'Excellent service and the product exceeded my expectations.',
    'I love this item, it is well made and very durable.',
    'Fantastic purchase, very happy with the quality and price.',
    'Outstanding product, would definitely buy again without hesitation.',
    'Very satisfied with this purchase, great value for money.',
    'Works exactly as described, super happy with this buy.',
    'The best purchase I have made this year, truly excellent.',
    'Brilliant product, does exactly what it promises and more.',
]
negative_templates = [
    'This product is terrible and completely broke after one day.',
    'Very disappointed, the quality is poor and not worth the price.',
    'Awful experience, the item arrived damaged and unusable.',
    'Worst purchase ever, do not waste your money on this.',
    'Completely useless product, stopped working within a week.',
    'Very bad quality, cheaply made and fell apart immediately.',
    'Do not buy this, it is a scam and not as advertised.',
    'Terrible customer service and the product is defective.',
    'I regret buying this, total waste of money and time.',
    'Horrible item, returned it immediately after seeing the quality.',
]
neutral_templates = [
    'The product is okay, nothing special but does the job.',
    'Average quality, meets basic requirements but could be better.',
    'It works as expected, neither great nor terrible overall.',
    'Decent product for the price, has some pros and cons.',
    'Standard item, exactly what was shown in the description.',
    'Not bad but not great either, a fairly average purchase.',
    'It is fine, does what it needs to do, nothing more.',
    'Satisfactory product, arrived on time and works normally.',
    'Mediocre quality but acceptable given the low price point.',
    'Neither impressed nor disappointed, just an ordinary item.',
]

n_per_class = 400
reviews, labels = [], []
for templates, label in [(positive_templates, 'positive'), (negative_templates, 'negative'), (neutral_templates, 'neutral')]:
    for _ in range(n_per_class):
        base = np.random.choice(templates)
        variation = np.random.choice(['', ' Really.', ' Overall good.', ' Would recommend.', ' Think twice.'])
        reviews.append(base + variation)
        labels.append(label)

df = pd.DataFrame({'review': reviews, 'sentiment': labels}).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Dataset shape: {df.shape}')
print(df['sentiment'].value_counts())
df.head()

## Task 2: Text Preprocessing

In [ ]:
STOPWORDS = {'i','me','my','we','our','the','a','an','and','or','but','is','it','its','this','that','to','of','in','for','on','with','as','at','by','from','be','was','are','has','have','had','do','does','did','so','if'}

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(preprocess_text)
print('Preprocessing done.')
print('Before:', df['review'][0])
print('After: ', df['cleaned_review'][0])

## Task 3: Text Vectorization

In [ ]:
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

X = df['cleaned_review']
y = df['label']

X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train_text)
X_test_bow = bow.transform(X_test_text)

print(f'TF-IDF matrix: {X_train_tfidf.shape}')
print(f'BoW matrix: {X_train_bow.shape}')

## Task 4: Baseline Models

In [ ]:
lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression (TF-IDF) Accuracy: {lr_acc:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Neutral', 'Positive']))

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)
y_pred_nb = nb_model.predict(X_test_bow)
nb_acc = accuracy_score(y_test, y_pred_nb)
print(f'Naive Bayes (BoW) Accuracy: {nb_acc:.4f}')

## Task 5: LSTM Sequence Model

In [ ]:
MAX_WORDS = 5000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN)

lstm_model = keras.Sequential([
    layers.Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

history = lstm_model.fit(X_train_pad, y_train, validation_data=(X_test_pad, y_test), epochs=10, batch_size=32, verbose=1)
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_pad, y_test, verbose=0)
print(f'LSTM Test Accuracy: {lstm_acc:.4f}')

## Task 6: Attention and Transformer Reflection

### Why RNNs Struggle with Long-Term Dependencies
Standard RNNs process sequences step-by-step, maintaining a hidden state. However, gradients vanish or explode as they propagate through many time steps. This means information from early tokens in a long sequence becomes diluted.

### How LSTMs Help with Memory
LSTMs introduce a **cell state** and three gating mechanisms:
- **Forget gate:** Decides what to remove from memory
- **Input gate:** Decides what new information to store
- **Output gate:** Decides what to output from memory

### What Attention Solves in Sequence-to-Sequence Tasks
In tasks like machine translation, the encoder must compress an entire source sentence into a fixed-size vector — a bottleneck. The **attention mechanism** lets the decoder look back at all encoder hidden states and dynamically focus on the most relevant parts at each decoding step.

### Why Transformers Are Important in Modern NLP
Transformers replace recurrence entirely with **self-attention**, enabling:
- **Parallelization:** All positions processed simultaneously
- **Global context:** Every token attends to every other token directly
- **Scalability:** Scale to billions of parameters
- **Transfer learning:** Pre-trained transformers fine-tune effectively on downstream tasks

Transformers are the backbone of all modern Generative AI models: ChatGPT, Claude, Gemini, BERT, T5.